In [7]:
def encrypt(text, key):
    result = ""
    key %= 26

    for ch in text:
        if ch.isalpha():
            base = ord('A') if ch.isupper() else ord('a')
            result += chr((ord(ch) - base + key) % 26 + base)
        else:
            result += ch

    return result


def decrypt(text, key):
    result = ""
    key %= 26

    for ch in text:
        if ch.isalpha():
            base = ord('A') if ch.isupper() else ord('a')
            result += chr((ord(ch) - base - key) % 26 + base)
        else:
            result += ch

    return result


key = 0

while True:
    print("\n====== Caesar Cipher ======")
    print("---------------------------")
    print("1. Send a Message")
    print("2. Receive a Message")
    print(f"3. Change Key [Current Key: {key}]")
    print("0. Exit")

    choice = input("Enter your choice: ").strip()

    if choice == '1':
        plaintext = input("Enter plaintext: ").strip()
        if plaintext:
            cipher = encrypt(plaintext, key)
            print("Encrypted text:", cipher)
        else:
            print("Empty input!")

    elif choice == '2':
        cipher_input = input("Enter ciphertext: ").strip()
        if cipher_input:
            decrypted = decrypt(cipher_input, key)
            print("Decrypted text:", decrypted)
        else:
            print("Empty input!")

    elif choice == '3':
        try:
            key = int(input("Enter new key: "))
            print("Key changed successfully!")
        except ValueError:
            print("Invalid key! Please enter an integer.")

    elif choice == '0':
        print("Exiting program...")
        break

    else:
        print("Invalid choice! Try again.")



====== Caesar Cipher ======
---------------------------
1. Send a Message
2. Receive a Message
3. Change Key [Current Key: 0]
0. Exit
Exiting program...


In [8]:
def generate_key_matrix(key):
    key = key.upper().replace("J", "I")
    seen = set()
    matrix_list = []

    for ch in key:
        if ch.isalpha() and ch not in seen:
            seen.add(ch)
            matrix_list.append(ch)

    for ch in "ABCDEFGHIKLMNOPQRSTUVWXYZ":
        if ch not in seen:
            seen.add(ch)
            matrix_list.append(ch)

    return [matrix_list[i:i+5] for i in range(0, 25, 5)]


def find_position(matrix, ch):
    ch = ch.upper().replace("J", "I")

    for i in range(5):
        for j in range(5):
            if matrix[i][j] == ch:
                return i, j


def prepare_text(text):
    text = text.upper().replace("J", "I")
    text = "".join(ch for ch in text if ch.isalpha())

    prepared = ""
    i = 0

    while i < len(text):
        a = text[i]
        b = text[i + 1] if i + 1 < len(text) else 'X'

        if a == b:
            prepared += a + 'X'
            i += 1
        else:
            prepared += a + b
            i += 2

    if len(prepared) % 2 != 0:
        prepared += 'X'

    return prepared


def encrypt_playfair(text, matrix):
    text = prepare_text(text)
    cipher = ""

    for i in range(0, len(text), 2):
        a, b = text[i], text[i + 1]
        r1, c1 = find_position(matrix, a)
        r2, c2 = find_position(matrix, b)

        if r1 == r2:
            cipher += matrix[r1][(c1 + 1) % 5]
            cipher += matrix[r2][(c2 + 1) % 5]

        elif c1 == c2:
            cipher += matrix[(r1 + 1) % 5][c1]
            cipher += matrix[(r2 + 1) % 5][c2]

        else:
            cipher += matrix[r1][c2]
            cipher += matrix[r2][c1]

    return cipher


def decrypt_playfair(text, matrix):
    text = "".join(ch for ch in text.upper().replace("J", "I") if ch.isalpha())

    if len(text) % 2 != 0:
        text += "X"

    plain = ""

    for i in range(0, len(text), 2):
        a, b = text[i], text[i + 1]
        r1, c1 = find_position(matrix, a)
        r2, c2 = find_position(matrix, b)

        if r1 == r2:
            plain += matrix[r1][(c1 - 1) % 5]
            plain += matrix[r2][(c2 - 1) % 5]

        elif c1 == c2:
            plain += matrix[(r1 - 1) % 5][c1]
            plain += matrix[(r2 - 1) % 5][c2]

        else:
            plain += matrix[r1][c2]
            plain += matrix[r2][c1]

    return plain


keyword = "MONARCHY"
matrix = generate_key_matrix(keyword)

while True:
    print("\n====== Playfair Cipher ======")
    print("-----------------------------")
    print("1. Send a Message")
    print("2. Receive a Message")
    print(f"3. Change Keyword [Current Keyword: {keyword}]")
    print("4. Show Key Matrix")
    print("0. Exit")

    choice = input("Enter your choice: ").strip()

    if choice == '1':
        plaintext = input("Enter plaintext: ").strip()

        if plaintext:
            cipher = encrypt_playfair(plaintext, matrix)
            print("Encrypted text:", cipher)
        else:
            print("Empty input!")

    elif choice == '2':
        cipher_input = input("Enter ciphertext: ").strip()

        if cipher_input:
            decrypted = decrypt_playfair(cipher_input, matrix)
            print("Decrypted text:", decrypted)
        else:
            print("Empty input!")

    elif choice == '3':
        new_keyword = input("Enter new keyword: ").strip()

        if new_keyword and any(ch.isalpha() for ch in new_keyword):
            keyword = new_keyword.upper().replace("J", "I")
            matrix = generate_key_matrix(keyword)
            print("Keyword changed successfully!")
        else:
            print("Invalid keyword! Please enter alphabetic characters.")

    elif choice == '4':
        print("Key Matrix:")
        for row in matrix:
            print(" ".join(row))

    elif choice == '0':
        print("Exiting program...")
        break

    else:
        print("Invalid choice! Try again.")


====== Playfair Cipher ======
-----------------------------
1. Send a Message
2. Receive a Message
3. Change Keyword [Current Keyword: MONARCHY]
4. Show Key Matrix
0. Exit
Exiting program...


In [9]:
def mod_inverse(a, m):
    a %= m
    for x in range(1, m):
        if (a * x) % m == 1:
            return x
    raise ValueError("No modular inverse exists.")


def matrix_inverse_2x2(matrix):
    a, b = matrix[0]
    c, d = matrix[1]

    det = (a * d - b * c) % 26
    det_inv = mod_inverse(det, 26)

    inv = [
        [(d * det_inv) % 26, (-b * det_inv) % 26],
        [(-c * det_inv) % 26, (a * det_inv) % 26]
    ]

    return inv


def is_valid_key(matrix):
    try:
        matrix_inverse_2x2(matrix)
        return True
    except ValueError:
        return False


def process_text(text):
    text = "".join(ch for ch in text.upper() if ch.isalpha())

    if len(text) % 2 != 0:
        text += 'X'

    return text


def text_to_numbers(text):
    return [ord(ch) - ord('A') for ch in text]


def numbers_to_text(nums):
    return "".join(chr(n % 26 + ord('A')) for n in nums)


def encrypt_hill(text, key):
    text = process_text(text)
    nums = text_to_numbers(text)
    result = []

    for i in range(0, len(nums), 2):
        pair = nums[i:i + 2]

        c1 = (key[0][0] * pair[0] + key[0][1] * pair[1]) % 26
        c2 = (key[1][0] * pair[0] + key[1][1] * pair[1]) % 26

        result.extend([c1, c2])

    return numbers_to_text(result)


def decrypt_hill(cipher, key):
    cipher = process_text(cipher)
    inv_key = matrix_inverse_2x2(key)
    nums = text_to_numbers(cipher)
    result = []

    for i in range(0, len(nums), 2):
        pair = nums[i:i + 2]

        p1 = (inv_key[0][0] * pair[0] + inv_key[0][1] * pair[1]) % 26
        p2 = (inv_key[1][0] * pair[0] + inv_key[1][1] * pair[1]) % 26

        result.extend([p1, p2])

    return numbers_to_text(result)


def show_key_matrix(key):
    print("Key Matrix:")
    for row in key:
        print(row)


def input_key_matrix():
    try:
        print("Enter 2x2 key matrix values:")
        a = int(input("Enter value a: "))
        b = int(input("Enter value b: "))
        c = int(input("Enter value c: "))
        d = int(input("Enter value d: "))

        new_key = [[a, b], [c, d]]

        if is_valid_key(new_key):
            return new_key
        else:
            print("Invalid key matrix! Determinant has no inverse modulo 26.")
            return None

    except ValueError:
        print("Invalid input! Please enter integer values.")
        return None


key = [[3, 3], [2, 5]]

while True:
    print("\n====== Hill Cipher ======")
    print("-------------------------")
    print("1. Send a Message")
    print("2. Receive a Message")
    print(f"3. Change Key Matrix [Current Key: {key}]")
    print("4. Show Key Matrix")
    print("0. Exit")

    choice = input("Enter your choice: ").strip()

    if choice == '1':
        plaintext = input("Enter plaintext: ").strip()

        if plaintext:
            cipher = encrypt_hill(plaintext, key)
            print("Encrypted text:", cipher)
        else:
            print("Empty input!")

    elif choice == '2':
        cipher_input = input("Enter ciphertext: ").strip()

        if cipher_input:
            decrypted = decrypt_hill(cipher_input, key)
            print("Decrypted text:", decrypted)
        else:
            print("Empty input!")

    elif choice == '3':
        new_key = input_key_matrix()

        if new_key:
            key = new_key
            print("Key matrix changed successfully!")

    elif choice == '4':
        show_key_matrix(key)

    elif choice == '0':
        print("Exiting program...")
        break

    else:
        print("Invalid choice! Try again.")


====== Hill Cipher ======
-------------------------
1. Send a Message
2. Receive a Message
3. Change Key Matrix [Current Key: [[3, 3], [2, 5]]]
4. Show Key Matrix
0. Exit
Exiting program...


In [10]:
def generate_key(text, key):
    key = key.upper()
    text = text.upper()
    result = ""
    j = 0

    for ch in text:
        if ch.isalpha():
            result += key[j % len(key)]
            j += 1
        else:
            result += ch

    return result


def encrypt_vigenere(text, key):
    text = text.upper()
    key = generate_key(text, key)
    cipher = ""

    for t, k in zip(text, key):
        if t.isalpha():
            cipher += chr((ord(t) - ord('A') + ord(k) - ord('A')) % 26 + ord('A'))
        else:
            cipher += t

    return cipher


def decrypt_vigenere(cipher, key):
    cipher = cipher.upper()
    key = generate_key(cipher, key)
    plain = ""

    for c, k in zip(cipher, key):
        if c.isalpha():
            plain += chr((ord(c) - ord('A') - (ord(k) - ord('A'))) % 26 + ord('A'))
        else:
            plain += c

    return plain


keyword = "KEY"

while True:
    print("\n====== Vigenere Cipher ======")
    print("-----------------------------")
    print("1. Send a Message")
    print("2. Receive a Message")
    print(f"3. Change Keyword [Current Keyword: {keyword}]")
    print("0. Exit")

    choice = input("Enter your choice: ").strip()

    if choice == '1':
        plaintext = input("Enter plaintext: ").strip()
        if plaintext:
            cipher = encrypt_vigenere(plaintext, keyword)
            print("Encrypted text:", cipher)
        else:
            print("Empty input!")

    elif choice == '2':
        cipher_input = input("Enter ciphertext: ").strip()
        if cipher_input:
            decrypted = decrypt_vigenere(cipher_input, keyword)
            print("Decrypted text:", decrypted)
        else:
            print("Empty input!")

    elif choice == '3':
        new_keyword = input("Enter new keyword: ").strip()

        if new_keyword.isalpha():
            keyword = new_keyword.upper()
            print("Keyword changed successfully!")
        else:
            print("Invalid keyword! Please enter alphabetic characters only.")

    elif choice == '0':
        print("Exiting program...")
        break

    else:
        print("Invalid choice! Try again.")


====== Vigenere Cipher ======
-----------------------------
1. Send a Message
2. Receive a Message
3. Change Keyword [Current Keyword: KEY]
0. Exit
Exiting program...


In [11]:
def encrypt_rail_fence(text, key):
    if key <= 1:
        return text

    rail = ['' for _ in range(key)]
    row = 0
    direction = 1

    for ch in text:
        rail[row] += ch
        row += direction

        if row == 0 or row == key - 1:
            direction *= -1

    return ''.join(rail)


def decrypt_rail_fence(cipher, key):
    if key <= 1:
        return cipher

    pattern = [['\n' for _ in range(len(cipher))] for _ in range(key)]

    row, direction = 0, 1
    for col in range(len(cipher)):
        pattern[row][col] = '*'
        row += direction

        if row == 0 or row == key - 1:
            direction *= -1

    index = 0
    for i in range(key):
        for j in range(len(cipher)):
            if pattern[i][j] == '*' and index < len(cipher):
                pattern[i][j] = cipher[index]
                index += 1

    result = []
    row, direction = 0, 1
    for col in range(len(cipher)):
        result.append(pattern[row][col])
        row += direction

        if row == 0 or row == key - 1:
            direction *= -1

    return ''.join(result)


depth = 3

while True:
    print("\n====== Rail Fence Cipher ======")
    print("-------------------------------")
    print("1. Send a Message")
    print("2. Receive a Message")
    print(f"3. Change Depth [Current Depth: {depth}]")
    print("0. Exit")

    choice = input("Enter your choice: ").strip()

    if choice == '1':
        plaintext = input("Enter plaintext: ").strip()

        if plaintext:
            cipher = encrypt_rail_fence(plaintext, depth)
            print("Encrypted text:", cipher)
        else:
            print("Empty input!")

    elif choice == '2':
        cipher_input = input("Enter ciphertext: ").strip()

        if cipher_input:
            decrypted = decrypt_rail_fence(cipher_input, depth)
            print("Decrypted text:", decrypted)
        else:
            print("Empty input!")

    elif choice == '3':
        try:
            new_depth = int(input("Enter new depth: "))

            if new_depth > 1:
                depth = new_depth
                print("Depth changed successfully!")
            else:
                print("Invalid depth! Please enter a value greater than 1.")

        except ValueError:
            print("Invalid depth! Please enter an integer.")

    elif choice == '0':
        print("Exiting program...")
        break

    else:
        print("Invalid choice! Try again.")


====== Rail Fence Cipher ======
-------------------------------
1. Send a Message
2. Receive a Message
3. Change Depth [Current Depth: 3]
0. Exit
Exiting program...


In [12]:
from math import gcd

def mod_inverse(e, phi):
    for d in range(1, phi):
        if (d * e) % phi == 1:
            return d
    raise ValueError("Modular inverse not found.")

def rsa_keygen(p, q, e):
    n = p * q
    phi = (p - 1) * (q - 1)

    if gcd(e, phi) != 1:
        raise ValueError("e must be coprime with phi(n).")

    d = mod_inverse(e, phi)
    return (e, n), (d, n)

def rsa_encrypt(text, public_key):
    e, n = public_key
    return [pow(ord(ch), e, n) for ch in text]

def rsa_decrypt(cipher, private_key):
    d, n = private_key
    return ''.join(chr(pow(c, d, n)) for c in cipher)

# Fixed primes
p = 61
q = 53

# initial e
e = 17

def generate_keys():
    return rsa_keygen(p, q, e)

public_key, private_key = generate_keys()

while True:
    print("\n====== RSA MENU ======")
    print("Public Key :", public_key)
    print("Private Key:", private_key)
    print("----------------------")
    print("1. Send a Message")
    print("2. Receive a Message")
    print("3. Change Public Key (e)")
    print("0. Exit")

    choice = input("Enter your choice: ")

    if choice == '1':
        plaintext = input("Enter plaintext: ")
        cipher = rsa_encrypt(plaintext, public_key)
        print("Encrypted text:", cipher)

        decrypted = rsa_decrypt(cipher, private_key)
        print("Decrypted text:", decrypted)

    elif choice == '2':
        cipher_input = input("Enter Ciphertext (space separated numbers): ")
        try:
            cipher = list(map(int, cipher_input.split()))
            decrypted = rsa_decrypt(cipher, private_key)
            print("Decrypted text:", decrypted)
        except:
            print("Invalid ciphertext input!")

    elif choice == '3':
        try:
            new_e = int(input("Enter new value for e: "))
            phi = (p - 1) * (q - 1)

            if gcd(new_e, phi) != 1:
                print("Invalid e! It must be coprime with phi(n).")
            else:
                e = new_e
                public_key, private_key = generate_keys()
                print("Public key updated successfully!")

        except:
            print("Invalid input! Enter an integer.")

    elif choice == '0':
        print("Exiting program...")
        break

    else:
        print("Invalid choice! Try again.")


====== RSA MENU ======
Public Key : (17, 3233)
Private Key: (2753, 3233)
----------------------
1. Send a Message
2. Receive a Message
3. Change Public Key (e)
0. Exit
Exiting program...


In [13]:
def diffie_hellman(p, g, a, b):
    A = pow(g, a, p)
    B = pow(g, b, p)

    shared_A = pow(B, a, p)
    shared_B = pow(A, b, p)

    return A, B, shared_A, shared_B


# Default values
p = 23
g = 5
a = 6
b = 15

while True:
    print("\n====== Diffie-Hellman Key Exchange ======")
    print("------------------------------------------")
    print("1. Generate Keys")
    print("2. Change Public Values (p, g)")
    print("3. Change Private Keys (a, b)")
    print(f"4. Show Current Values [p={p}, g={g}, a={a}, b={b}]")
    print("0. Exit")

    choice = input("Enter your choice: ").strip()

    if choice == '1':
        try:
            A, B, shared_A, shared_B = diffie_hellman(p, g, a, b)

            print("\nPublic key of A:", A)
            print("Public key of B:", B)
            print("Shared key for A:", shared_A)
            print("Shared key for B:", shared_B)

            if shared_A == shared_B:
                print("Shared secret key established successfully!")
            else:
                print("Error: Keys do not match!")

        except Exception as e:
            print("Error:", e)

    elif choice == '2':
        try:
            p = int(input("Enter prime number p: "))
            g = int(input("Enter primitive root g: "))
            print("Public values updated!")
        except ValueError:
            print("Invalid input! Enter integers only.")

    elif choice == '3':
        try:
            a = int(input("Enter private key of A: "))
            b = int(input("Enter private key of B: "))
            print("Private keys updated!")
        except ValueError:
            print("Invalid input! Enter integers only.")

    elif choice == '4':
        print(f"Current values → p={p}, g={g}, a={a}, b={b}")

    elif choice == '0':
        print("Exiting program...")
        break

    else:
        print("Invalid choice! Try again.")


====== Diffie-Hellman Key Exchange ======
------------------------------------------
1. Generate Keys
2. Change Public Values (p, g)
3. Change Private Keys (a, b)
4. Show Current Values [p=23, g=5, a=6, b=15]
0. Exit
Exiting program...
